# 03 — nn.Module, Layers & Activations

`nn.Module` is the base class for all neural network components in PyTorch. Every model, layer, and even loss function can subclass it.

## Defining a Model

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class MLP(nn.Module):
    def __init__(self, in_features, hidden, out_features):
        super().__init__()
        self.fc1 = nn.Linear(in_features, hidden)
        self.fc2 = nn.Linear(hidden, out_features)
        self.dropout = nn.Dropout(0.2)

    def forward(self, x):
        x = F.relu(self.fc1(x))
        x = self.dropout(x)
        return self.fc2(x)

model = MLP(784, 128, 10)
x = torch.randn(32, 784)
out = model(x)
out.shape   # (32, 10)


## nn.Sequential (for simple stacks)

In [ ]:
model = nn.Sequential(
    nn.Linear(784, 128),
    nn.ReLU(),
    nn.Dropout(0.2),
    nn.Linear(128, 10),
)
model(torch.randn(32, 784)).shape


## Common Layers

In [ ]:
nn.Linear(in_features=64, out_features=32)
nn.Conv1d(in_channels=1, out_channels=16, kernel_size=3)
nn.Conv2d(in_channels=3, out_channels=16, kernel_size=3, stride=1, padding=1)
nn.MaxPool2d(kernel_size=2)
nn.AvgPool2d(kernel_size=2)
nn.AdaptiveAvgPool2d(output_size=(1, 1))
nn.BatchNorm1d(num_features=32)
nn.BatchNorm2d(num_features=16)
nn.LayerNorm(normalized_shape=64)
nn.Dropout(p=0.5)
nn.Dropout2d(p=0.5)
nn.Embedding(num_embeddings=10000, embedding_dim=128)
nn.Flatten()


## Activations

In [ ]:
nn.ReLU()
nn.LeakyReLU(negative_slope=0.01)
nn.Sigmoid()
nn.Tanh()
nn.GELU()
nn.SiLU()          # aka Swish
nn.Softmax(dim=1)
nn.LogSoftmax(dim=1)

# functional equivalents (stateless, used inline in forward())
F.relu(x)
F.gelu(x)
F.softmax(x, dim=1)


## Parameters & State

In [ ]:
model = MLP(784, 128, 10)

for name, param in model.named_parameters():
    print(name, param.shape, param.requires_grad)

sum(p.numel() for p in model.parameters())              # total params
sum(p.numel() for p in model.parameters() if p.requires_grad)  # trainable params

model.state_dict().keys()      # ordered dict of all weights/buffers
model.named_modules()          # iterate submodules
model.children()               # immediate children only


## Freezing Parameters

In [ ]:
for param in model.fc1.parameters():
    param.requires_grad = False   # freeze this layer

# only fc2 will be updated by the optimizer
trainable = filter(lambda p: p.requires_grad, model.parameters())
optimizer = torch.optim.Adam(trainable, lr=1e-3)


## train() vs eval() mode

Matters for layers like `Dropout` and `BatchNorm` that behave differently at train vs inference time.

In [ ]:
model.train()   # dropout active, batchnorm uses batch stats
model.eval()    # dropout disabled, batchnorm uses running stats

with torch.no_grad():
    model.eval()
    preds = model(x)   # inference: no grad tracking + eval-mode layers


## Custom Layer (subclassing nn.Module)

In [ ]:
class ResidualBlock(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.fc = nn.Linear(dim, dim)
        self.norm = nn.LayerNorm(dim)

    def forward(self, x):
        return self.norm(x + F.relu(self.fc(x)))

block = ResidualBlock(64)
block(torch.randn(8, 64)).shape


## Weight Initialization

In [ ]:
def init_weights(m):
    if isinstance(m, nn.Linear):
        nn.init.kaiming_normal_(m.weight, nonlinearity="relu")
        nn.init.zeros_(m.bias)

model.apply(init_weights)   # recursively applies to all submodules


## Common Gotchas
- Forgetting `model.eval()` before inference silently keeps dropout/batchnorm in train mode.
- Layers must be assigned as attributes (`self.fc = ...`) — storing them in a plain Python list won't register their parameters (use `nn.ModuleList` instead).
- `nn.Sequential` requires each layer's output shape to match the next layer's expected input.

In [ ]:
class Net(nn.Module):
    def __init__(self):
        super().__init__()
        self.layers = nn.ModuleList([nn.Linear(10, 10) for _ in range(3)])  # correctly registered

    def forward(self, x):
        for layer in self.layers:
            x = F.relu(layer(x))
        return x
